# Cross-sectional momentum: z-scored 20-day returns

Quick research notebook. Idea: rank stocks on 20-day momentum, normalise each ticker's signal, hold the top decile with equal weights.

*Deliberately leaky demo for Backtest Auditor: it looks fine on a skim and backtests beautifully.*

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.width", 120)

## Data

Use the vendor CSV when it is there, otherwise fall back to the synthetic panel that ships with the repo.

In [2]:
from pathlib import Path

DATA_PATH = Path("data/prices.csv")
if DATA_PATH.exists():
    prices = (pd.read_csv(DATA_PATH, parse_dates=["date"])
                .pivot(index="date", columns="ticker", values="close"))
else:
    from data import generate_panel
    prices = generate_panel()["close"]
prices.shape

## Exploration

In [3]:
!pip list 2>/dev/null | grep -i pandas
rets = prices.pct_change()
rets.describe().T.head()

In [4]:
rets.mean().sort_values().plot(kind="bar", figsize=(12, 3), title="Mean daily return by ticker")
plt.show()

## Signal

Two helpers: a 20-day momentum and a per-ticker z-score so that high- and low-volatility names are comparable.

In [5]:
LOOKBACK = 20
TOP_PCT = 0.9


def zscore(df):
    """Normalise each ticker's series to mean 0, std 1."""
    return (df - df.mean()) / df.std()

In [6]:
def momentum(close, lookback=LOOKBACK):
    # trailing return, aligned with the day we trade on
    mom = close / close.shift(lookback) - 1
    return mom.shift(-1)

In [7]:
def generate_signals(data):
    """Z-scored 20-day momentum, top decile, equal weight, daily rebalance."""
    close = data["close"]
    score = zscore(momentum(close))
    rank = score.rank(axis=1, pct=True)
    top = (rank >= TOP_PCT).astype(float)
    return top.div(top.sum(axis=1).replace(0, np.nan), axis=0).fillna(0.0)

In [8]:
%%time
w = generate_signals({"close": prices})
w.sum(axis=1).describe()

In [9]:
strat = (w.shift(1) * rets).sum(axis=1)
print(f"Sharpe: {strat.mean() / strat.std() * np.sqrt(252):.2f}")
strat.cumsum().plot(title="Cumulative return")

## Conclusion

Sharpe well above 3 with tiny drawdowns. Ship it? (Run `python audit_cli.py examples/leaky_research.ipynb` first.)